### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Fri Jun 19 01:20:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   55C    P0             48W /  300W |       1MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/FaceAgePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/FaceAgeDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0
...,...,...,...
9145,475367_1941-08-03_2011.jpg,70,1.0
9146,304085_1919-07-07_1989.jpg,70,1.0
9147,nm0001627_rm4164078592_1927-2-20_1997.jpg,70,1.0
9148,nm0000024_rm1715129344_1904-4-14_1974.jpg,70,1.0


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)

9150


In [8]:
print(len(all_pc_faceage))

250249


In [10]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [11]:
crowd_labels.head()

,left,right,label,performer
0,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
1,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
2,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
3,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,1
4,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,1


In [12]:
from pathlib import Path

# Map item -> score
score_dict = dict(zip(df_faceage["full_path"].apply(lambda x: Path(x).name),
                      df_faceage["score"]))

left_correct = 0
right_correct = 0
ties = 0
total = 0

for _, row in crowd_labels.iterrows():
    left = Path(row["left"]).name
    right = Path(row["right"]).name

    if left not in score_dict or right not in score_dict:
        print('here')
        continue  # skip if missing

    total += 1
    left_score = score_dict[left]
    right_score = score_dict[right]

    if left_score > right_score:
        left_correct += 1
    elif right_score > left_score:
        right_correct += 1
    else:
        ties += 1

print(f"Total comparisons: {total}")
print(f"Left is true winner: {left_correct} ({left_correct/total:.3f})")
print(f"Right is true winner: {right_correct} ({right_correct/total:.3f})")
print(f"Ties: {ties} ({ties/total:.3f})")

Total comparisons: 250249
Left is true winner: 122835 (0.491)
Right is true winner: 123229 (0.492)
Ties: 4185 (0.017)


## Hyperparameter Setup

In [ ]:
max_iter=1000
lr = 0.01

### HTCV

In [10]:
import random
from htcv_m import *

In [11]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
gradem_time = []
for sd in range(1):
    start = time.time()
    grad_em = HTCVWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:05<00:00, 189.34it/s]



Reached max_epochs without full convergence.


In [12]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

7.573662757873535 +- 0.0


In [13]:
print(f"GradEM -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

GradEM -- Accuracy: 0.791631817817688 ± 0.0, Weighted Accuracy: 0.8732313513755798 ± 0.0, Kendall's Tau: 0.578494669088338 ± 0.0


### Gradient EM

In [14]:
import random
from grad_em import *

In [15]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
lr = 0.01
gradem_time = []
for sd in range(1):
    start = time.time()
    grad_em = GradientEMWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:04<00:00, 221.69it/s]



Reached max_epochs without full convergence.


In [16]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

5.365476608276367 +- 0.0


In [17]:
print(f"GradEM -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

GradEM -- Accuracy: 0.7914183735847473 ± 0.0, Weighted Accuracy: 0.8729842305183411 ± 0.0, Kendall's Tau: 0.5780712983069831 ± 0.0


### PG EM - flipped orders

In [26]:
import random
from pgem_initialize_beta_1 import EMWrapper_random_r

In [27]:
import time
max_iter = 200

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper_random_r(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  0%|▍                                                                                               | 1/200 [00:01<03:34,  1.08s/it]

Iter 000: Log-likelihood = -0.347991


 16%|██████████████▋                                                                                | 31/200 [00:50<04:36,  1.64s/it]

Converged at iter 31, ΔLL = 3.874302e-07
cuda


...

Converged at iter 31, ΔLL = 4.172325e-07
cuda



  0%|▍                                                                                               | 1/200 [00:01<03:39,  1.10s/it]

Iter 000: Log-likelihood = -0.347998


 16%|██████████████▋                                                                                | 31/200 [00:50<04:35,  1.63s/it]

Converged at iter 31, ΔLL = 3.874302e-07


In [28]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

40.288419818878175 +- 16.99191771688435


In [29]:
PGEM_accs

[0.7921145558357239,
 0.7921149134635925,
 0.7921150922775269,
 0.7921147346496582,
 0.792114794254303,
 0.7921149134635925,
 0.7921150922775269,
 0.7921145558357239,
 0.7921150922775269,
 0.792114794254303]

In [30]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7921148538589478 ± 1.9405866379856712e-07, Weighted Accuracy: 0.8736116766929627 ± 2.941837949574725e-07, Kendall's Tau: 0.5794528541398647 ± 4.0280739402059446e-07


### PG EM - flipped orders (r then beta)

In [10]:
import random
from pgem_initialize_beta_1 import EMWrapper_random_r

In [11]:
import time
max_iter = 200

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper_random_r(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  2%|█▍                                                                                              | 3/200 [00:01<01:10,  2.78it/s]

Iter 000: Log-likelihood = -0.349581


 38%|████████████████████████████████████                                                           | 76/200 [01:13<02:00,  1.03it/s]

Converged at iter 76, ΔLL = 9.834766e-07
cuda


...

Converged at iter 76, ΔLL = 9.834766e-07
cuda



  0%|▍                                                                                               | 1/200 [00:00<03:01,  1.10it/s]

Iter 000: Log-likelihood = -0.349576


 38%|████████████████████████████████████                                                           | 76/200 [01:02<01:42,  1.21it/s]

Converged at iter 76, ΔLL = 9.834766e-07


In [12]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

75.91461477279663 +- 8.311392954835778


In [13]:
PGEM_accs

[0.7921266555786133,
 0.7921267151832581,
 0.7921266555786133,
 0.7921265959739685,
 0.7921265959739685,
 0.7921267747879028,
 0.7921266555786133,
 0.7921267747879028,
 0.7921266555786133,
 0.7921265363693237]

In [14]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7921266615390777 ± 7.27567411406857e-08, Weighted Accuracy: 0.8736116766929627 ± 4.460403188197543e-08, Kendall's Tau: 0.5794762594301679 ± 1.5422991922044548e-07


### PG EM

In [18]:
import random
from pgem_initialize_beta_1 import EMWrapper, EMWrapper_3_0

In [19]:
max_iter = 200

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(1):
    start = time.time()
    pg = EMWrapper_3_0(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  0%|▍                                                                                  | 1/200 [00:01<04:05,  1.23s/it]

Iter 000: Log-likelihood = -0.366355


 38%|███████████████████████████████▌                                                  | 77/200 [02:09<03:27,  1.69s/it]

Converged at iter 77, ΔLL = 9.238720e-07


In [20]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

130.78954124450684 +- 0.0


In [21]:
PGEM_accs

[0.7921259999275208]

In [22]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7921259999275208 ± 0.0, Weighted Accuracy: 0.8736113905906677 ± 0.0, Kendall's Tau: 0.5794749717183667 ± 0.0


In [13]:
import time
max_iter = 100

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []

for sd in range(1):
    start = time.time()
    pg = EMWrapper_3_0(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    print("Elapsed:", end - start, "seconds")
    
    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cpu


  1%|▊                                                                                  | 1/100 [00:00<00:41,  2.38it/s]

Iter 000: Log-likelihood = -0.693147
Converged at iter 1, ΔLL = 0.000000e+00
Elapsed: 1.1214051246643066 seconds


In [14]:
PGEM_accs

[0.0]

In [19]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7920684814453125 ± 0.0, Weighted Accuracy: 0.8735842108726501 ± 0.0, Kendall's Tau: 0.5793608909265742 ± 0.0


### BT

In [20]:
%%time
bt_scores = choix.opt_pairwise(size, all_pc_faceage, alpha=0, method='Newton-CG', initial_params=None, max_iter=None, tol=1e-05)

CPU times: user 10min 28s, sys: 3min 16s, total: 13min 44s
Wall time: 1min 56s


In [21]:
r_est_np = to_numpy(bt_scores)
current_tau = safe_kendalltau(r_est_np, gt_scores)
if current_tau < 0:
    r_est_np = -r_est_np
bt_acc = compute_acc(df_faceage, 1*r_est_np, device)
bt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
bt_tau = safe_kendalltau(r_est_np, gt_scores)

In [22]:
print(f"Simple BT -- Accuracy: {bt_acc}, Weighted Accuracy: {bt_wacc}, Kendall's Tau: {bt_tau} ")

Simple BT -- Accuracy: 0.7900686264038086, Weighted Accuracy: 0.8719564080238342, Kendall's Tau: 0.575393885555222 


### BARP

In [10]:
classes = df_faceage['gender']
FaceAge = opt_fair.BARP(data = PC_faceage, penalty = 0, classes = classes, device=device)

In [11]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [12]:
%%time
annot_bt_temp, annot_bias =  opt_fair._alternate_optim_torch(size, num_reviewers, FaceAge, iters = 100)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [08:03<00:00,  4.83s/it]

CPU times: user 6min 1s, sys: 2min 3s, total: 8min 4s
Wall time: 8min 4s


In [13]:
r_est_np = to_numpy(annot_bt_temp)
gt_scores = to_numpy(df_faceage['score'].tolist())
current_tau = safe_kendalltau(r_est_np, gt_scores)
if current_tau < 0:
    r_est_np = -r_est_np
barp_acc = compute_acc(df_faceage, 1*r_est_np, device)
barp_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
barp_tau = safe_kendalltau(r_est_np, gt_scores)

In [14]:
print(f"BARP -- Accuracy: {barp_acc}, Weighted Accuracy: {barp_wacc}, Kendall's Tau: {barp_tau} ")

BARP -- Accuracy: 0.7911810278892517, Weighted Accuracy: 0.8728458285331726, Kendall's Tau: 0.5776003949914079 


### RC

In [28]:
%%time

rc_obj = RankCentrality(device)
A = rc_obj.matrix_of_comparisons(size, all_pc_faceage)
# print("A matrix done")
P = rc_obj.trans_prob(A)
# print("P matrix done")
pi = rc_obj.stationary_dist(P)
rank_centrality_scores = torch.log(pi).cpu().numpy()

CPU times: user 1.07 s, sys: 1.45 s, total: 2.52 s
Wall time: 2.52 s


In [29]:
r_est_np = to_numpy(rank_centrality_scores)
current_tau = safe_kendalltau(r_est_np, gt_scores)
if current_tau < 0:
    r_est_np = -r_est_np
rc_acc = compute_acc(df_faceage, 1*r_est_np, device)
rc_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
rc_tau = safe_kendalltau(r_est_np, gt_scores)

In [30]:
print(f"RC -- Accuracy: {rc_acc}, Weighted Accuracy: {rc_wacc}, Kendall's Tau: {rc_tau} ")

RC -- Accuracy: 0.7804011106491089, Weighted Accuracy: 0.8644054532051086, Kendall's Tau: 0.5582118883509699 


### CrowdBT

In [37]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [38]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())
import time

cuda


In [39]:
max_iter = 1000
CrowdBT_accs, CrowdBT_waccs, CrowdBT_taus = [], [], []
K = num_reviewers
gt_df = df_faceage
crowdbt_time = []
for seed in range(1):
    try:
        start = time.time()
        crowdbt_test = opt_fair.CrowdBT_3_0(data=PC_faceage, penalty=0, device=device, random_seed=seed)
        crowdbt_scores, _ = crowdbt_test.alternate_optim(size, K, lr_x=0.01, lr_y=0.01, iters=max_iter)
        end = time.time()
        crowdbt_time.append(end-start)
        r_est_np = to_numpy(crowdbt_scores)
        gt_scores = to_numpy(df_faceage['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        crowdbt_acc = compute_acc(df_faceage, 1*r_est_np, device)
        crowdbt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
        crowdbt_tau = safe_kendalltau(r_est_np, gt_scores)
        CrowdBT_accs.append(crowdbt_acc)
        CrowdBT_waccs.append(crowdbt_wacc)
        CrowdBT_taus.append(crowdbt_tau)
    except Exception as e:
        print(e)
        CrowdBT_accs.append(0.0)
        CrowdBT_waccs.append(0.0)
        CrowdBT_taus.append(0.0)

100%|█████████████████████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 91.99it/s, loss=6.74e+4]


In [40]:
print(f"{np.mean(crowdbt_time)} +- {np.std(crowdbt_time)}")

11.281454086303711 +- 0.0


In [41]:
print(f"CrowdBT -- Accuracy: {np.mean(CrowdBT_accs)} ± {np.std(CrowdBT_accs)}, Weighted Accuracy: {np.mean(CrowdBT_waccs)} ± {np.std(CrowdBT_waccs)}, Kendall's Tau: {np.mean(CrowdBT_taus)} ± {np.std(CrowdBT_taus)}")

CrowdBT -- Accuracy: 0.7914734482765198 ± 0.0, Weighted Accuracy: 0.8729157447814941 ± 0.0, Kendall's Tau: 0.578180561499076 ± 0.0


### FactorBT

In [30]:
from crowdkit.aggregation import NoisyBradleyTerry

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
def sort_df(df, column_name):
    # Sort by a specific column (replace 'column_name' with your column)
    df_sorted = df.sort_values(by=column_name, ascending=True)  # or ascending=False

    return df_sorted

df = pd.read_csv('data/crowd_labels.csv')
df = df.rename(columns={'performer': 'worker'})

agg_noisybt = NoisyBradleyTerry(n_iter=100).fit_predict(df)
agg_noisybt_df = pd.DataFrame(list(agg_noisybt.items()), columns=['label', 'score'])
agg_noisybt_df = sort_df(agg_noisybt_df, 'label')
factorbt_scores = list(agg_noisybt_df['score'])

In [32]:
gt_df = pd.read_csv("data/gt.csv")
gt_df = sort_df(gt_df, 'label')
gt_scores = to_numpy(gt_df['score'].tolist())
r_est_np = to_numpy(factorbt_scores)

current_tau = safe_kendalltau(r_est_np, gt_scores)
if current_tau < 0:
    r_est_np = -r_est_np
factorbt_acc = compute_acc(gt_df, 1*r_est_np, device)
factorbt_wacc = compute_weighted_acc(gt_df, 1*r_est_np, device)
factorbt_tau = safe_kendalltau(r_est_np, gt_scores)

In [33]:
print(f"FactorBT -- Accuracy: {factorbt_acc}, Weighted Accuracy: {factorbt_wacc}, Kendall's Tau: {factorbt_tau} ")

FactorBT -- Accuracy: 0.7904868125915527, Weighted Accuracy: 0.8285281658172607, Kendall's Tau: 0.5762237655043003 
